# 9. Semantic Schemas

This tutorial shows how to attach semantic schema data to a KItem using the SDK.
Semantic schemas are defined in the k-type spec and map a KItem's scientific content
to ontology-typed RDF nodes.  They are distinct from `custom_properties`, which drive
the UI form.

By the end of this tutorial you will know:

- what `custom_properties` and `schema_data` each represent,
- how to populate both at construction time using the same input style,
- how to use the `populate_schema()` method for post-construction updates.

## 9.1. Setting up

Before you run this tutorial: make sure to have access to a DSMS-instance of your
interest, along with installation of this package, and have established access to the
DSMS through DSMS-SDK (refer to
[Connecting to DSMS](../dsms_sdk.md#connecting-to-dsms)).

Import the needed classes and functions.

In [ ]:
import os
from dsms import DSMS, KItem

dsms = DSMS(env=".env") if os.path.exists(".env") else DSMS()

## 9.2. Custom properties vs. semantic schemas

A KItem can carry two complementary representations of its data:

| | `custom_properties` | `schema_data` |
|---|---|---|
| **Purpose** | UI form displayed on the platform | OO-LD / RDF graph for SPARQL and semantic reasoning |
| **Defined by** | K-type webform schema | K-type semantic schema spec (v2) |
| **Input format** | Flat dict keyed by field label | Flat dict keyed by transform field names |
| **Set at init?** | Yes | Yes |

Fields such as `name` appear in both representations and should be kept consistent.
Geometry and administrative metadata typically live in `custom_properties`;
measurement results, provenance, and ontology-typed relationships live in `schema_data`.

## 9.3. Discovering available semantic schemas

The k-type spec lists which semantic schemas a KItem of that type can carry.
Fetch the v2 spec for any k-type to see the available schema IDs.

In [ ]:
ktype_v2 = dsms.get_v2_ktype("tensile-test")
for s in ktype_v2.spec.resolved_semantic_schemas:
    print(s.id, "→", s.url)

## 9.4. Setting both at construction time

Both `custom_properties` and `schema_data` accept a plain dict at construction.
For `schema_data`, pass `{schema_id: simplified_input_dict}`. The schema transform
is fetched and applied immediately; subsequent constructions with the same schema
URL use a process-level cache, keeping the cost comparable to `custom_properties`.


In [ ]:
specimen = KItem(
    name="Specimen-TT-01",
    ktype_id=dsms.ktypes.Specimen,
    custom_properties={
        "Specimen type": "flat",
        "Width": 12.5,
        "Length": 80.0,
        "Thickness": 1.5,
    },
    schema_data={
        "specimen/PMDCo": {
            "label": "Specimen-TT-01",
            "width_mm": 12.5,
            "length_mm": 80.0,
            "thickness_mm": 1.5,
        }
    },
)

specimen

Commit to the platform.  The `schema_data` is already resolved to OO-LD
and will be persisted as-is.


In [ ]:
dsms.add(specimen)
dsms.commit()
specimen.url

The `schema_data` already holds the resolved OO-LD content right after
construction — no need to wait for a commit to inspect it.


In [ ]:
for entry in specimen.schema_data:
    print("schema_id:", entry.schema_id)
    print("content keys:", list(entry.content.keys()))

## 9.5. Using `populate_schema()` for post-construction updates

`populate_schema(schema_id, input_data)` is the method-call equivalent.  Use it
when you need to add or replace a schema entry after the KItem has been constructed,
or when you want explicit control over the transform step.

It returns `self`, so calls can be chained.

In [ ]:
tensile_test = KItem(
    name="TensileTest-01",
    ktype_id=dsms.ktypes.TensileTest,
    custom_properties={
        "Identifier": "TT-2024-001",
        "Start time": "2024-03-15T09:00:00",
        "End time": "2024-03-15T09:45:00",
    },
)

tensile_test.populate_schema(
    "characterization/tensile-test/TTO",
    {
        "test_name": "TT-2024-001",
        "specimen_iri": str(specimen.id),
        "results": [
            {"property": "YieldStrength", "value": 350.0, "unit": "MPa"},
            {"property": "UltimateTensileStrength", "value": 490.0, "unit": "MPa"},
            {"property": "Elongation", "value": 28.5, "unit": "%"},
        ],
    },
)

dsms.add(tensile_test)
dsms.commit()
tensile_test.url

## 9.6. Inspecting the RDF subgraph

The platform generates an RDF subgraph from `schema_data` asynchronously after
each commit.  Use `kitem.subgraph` to retrieve and inspect it.

In [ ]:
import time
time.sleep(3)  # allow server-side graph generation to complete

try:
    print(tensile_test.subgraph.serialize(format="turtle"))
except ValueError:
    print("Subgraph not yet available — retry in a moment.")

## 9.7. Updating a schema entry

Calling `populate_schema()` with the same `schema_id` replaces the existing entry.
The dict shorthand at construction works the same way.

In [ ]:
tensile_test.populate_schema(
    "characterization/tensile-test/TTO",
    {
        "test_name": "TT-2024-001",
        "specimen_iri": str(specimen.id),
        "results": [
            {"property": "YieldStrength", "value": 355.0, "unit": "MPa"},
            {"property": "UltimateTensileStrength", "value": 495.0, "unit": "MPa"},
            {"property": "Elongation", "value": 29.0, "unit": "%"},
        ],
    },
)

dsms.add(tensile_test)
dsms.commit()

## 9.8. Cleanup

In [ ]:
del dsms[tensile_test]
del dsms[specimen]
dsms.commit()